<a href="https://colab.research.google.com/github/Carlos-Biologist/Automated-_Detection_and_Tracking_Pinnipeds_UAVs_YOLO26_-_BoT-SORT/blob/main/Modelo_Julian_Mathias.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ======================================================
# CELL A. Configuration, reading images/labels and identifying the FLIGHT of each image
#   Flight identification (in this order, per image):
#     1) file name: DJI_YYYYMMDDhhmmss_NNNN (video/flight stamp) or any YYYYMMDD / YYYY-MM-DD date
#     2) EXIF date/time (photos): consecutive photos with a gap <= GAP_MIN minutes = same flight
#     3) visual similarity (colour thumbnails + histograms) for the images with neither of the above
#   The column 'fonte' of the output CSV says how each image was assigned, so you can audit it.
# ======================================================
import os, re, shutil, random
import numpy as np, pandas as pd
from PIL import Image

SRC_IMG = '/content/drive/MyDrive/Colab Notebooks/YOLO_IL_Negativo_ML/YOLO_v26_Negativo/Modelo_Unico_IL_ML/Modelo_Julian_Mathias/train/images'
SRC_LBL = '/content/drive/MyDrive/Colab Notebooks/YOLO_IL_Negativo_ML/YOLO_v26_Negativo/Modelo_Unico_IL_ML/Modelo_Julian_Mathias/train/labels'
OUT_DIR = '/content/drive/MyDrive/Colab Notebooks/YOLO_IL_Negativo_ML/YOLO_v26_Negativo/Modelo_Unico_IL_ML/Modelo_Julian_Mathias/Split_70_15_15_por_voo_2'   # created new; the source is never modified

PROP = {'train': 0.70, 'val': 0.15, 'test': 0.15}
AGRUPAR_POR = 'data'     # 'voo'  : each flight goes entirely to ONE split (flights of the same date may be in different splits)
                         # 'data' : all flights of the same date go together to ONE split (stricter against leakage)
GAP_MIN = 20             # minutes: EXIF photos separated by more than this belong to different flights
N_VOOS_VISUAL = 6        # only used if NO image has a date in the name or in the EXIF
OVERRIDE_CSV = None      # optional CSV with columns prefixo;voo to force the flight of files whose name starts with 'prefixo'
SEED = 42
COPIAR = True            # False = only computes and prints the split (nothing is copied)
CLASS_NAMES = ['A_AUSTRALIS', 'O_FLAVESCENS']
# ---- label schemas in the merged folder ----
#   IL (raw export)   : 0 = A_AUSTRALIS, 1 = M_LEONINA (EXCLUDED), 2 = O_FLAVESCENS
#   ML (final schema) : 0 = A_AUSTRALIS, 1 = O_FLAVESCENS
#   Output (both)     : 0 = A_AUSTRALIS, 1 = O_FLAVESCENS   (IL: class 2 -> 1, M. leonina boxes removed)
# Which schema does each image follow? Decided in this order:
#   1) its original name (without the Roboflow hash) exists in one of IL_ORIGEM_DIRS   -> IL
#   2) IL_NOME_REGEX / ML_NOME_REGEX match the file name                              -> IL / ML
#   3) the label file contains class 2                                                -> IL
#   4) only class 0 (or empty)                                                        -> same in both schemas
#   5) contains class 1 but no class 2 -> AMBIGUOUS (M. leonina of IL or O. flavescens of ML): see AMBIGUAS
IL_ORIGEM_DIRS = []        # optional: folders with the ORIGINAL IL images (any depth), e.g. the IL raw export
IL_NOME_REGEX  = None      # optional, e.g. r'(janeiro|fevereiro|marco|abril|maio|junho|julho|agosto|setembro|outubro|novembro|dezembro)-20\d\d'
ML_NOME_REGEX  = None      # optional, e.g. r'Revis-ML|WRML'
AMBIGUAS = 'ML'        # 'avisar' (stop and list) | 'IL' | 'ML'  -> schema assumed for the ambiguous images
SEM_TXT_COMO_NEGATIVA = 'nome'   # images without a .txt: True = all negatives | False = ignored | 'nome' = negatives only if the name starts with "negativo"
EXTS = ('.jpg', '.jpeg', '.png')

random.seed(SEED); np.random.seed(SEED)
assert abs(sum(PROP.values()) - 1) < 1e-9
assert os.path.isdir(SRC_IMG) and os.path.isdir(SRC_LBL), "Source folders not found"

# ---- 1. read images + labels (schema-aware) ----
RF = re.compile(r'[_\.]?(jpg|jpeg|png)\.rf\.[0-9a-f]+$', re.I)       # Roboflow suffix of augmented copies
il_bases = set()
for d_ in IL_ORIGEM_DIRS:
    for root_, _, fs_ in os.walk(d_):
        for f_ in fs_:
            if f_.lower().endswith(EXTS):
                il_bases.add(RF.sub('', os.path.splitext(f_)[0]))

rows, SEM_LABEL = [], []
for f in sorted(os.listdir(SRC_IMG)):
    if not f.lower().endswith(EXTS):
        continue
    stem = os.path.splitext(f)[0]
    base = RF.sub('', stem)
    lbl = os.path.join(SRC_LBL, stem + '.txt')
    cnt = {0: 0, 1: 0, 2: 0}
    has_txt = os.path.exists(lbl)
    if has_txt:
        for line in open(lbl):
            p = line.split()
            if not p:
                continue
            c = int(p[0])
            if c not in cnt:
                raise ValueError(f"Unexpected class {c} in {lbl}")
            cnt[c] += 1
    if not has_txt:
        if SEM_TXT_COMO_NEGATIVA is True or (SEM_TXT_COMO_NEGATIVA == 'nome' and f.lower().startswith('negativo')):
            pass                                   # treated as a negative
        else:
            SEM_LABEL.append(f); continue          # ignored (as in the single-area notebooks)
    if base in il_bases or (IL_NOME_REGEX and re.search(IL_NOME_REGEX, f, re.I)):
        schema = 'IL'
    elif ML_NOME_REGEX and re.search(ML_NOME_REGEX, f, re.I):
        schema = 'ML'
    elif cnt[2] > 0:
        schema = 'IL'
    elif cnt[1] > 0:
        schema = 'AMBIGUA'
    else:
        schema = 'neutra'
    rows.append({'image': f, 'stem': stem, 'base': base, 'src_img': os.path.join(SRC_IMG, f), 'src_lbl': lbl,
                 'c0': cnt[0], 'c1': cnt[1], 'c2': cnt[2], 'schema': schema, 'sem_txt': not has_txt})
df = pd.DataFrame(rows)
if SEM_LABEL:
    print(f"⚠️ {len(SEM_LABEL)} images without .txt IGNORED (SEM_TXT_COMO_NEGATIVA = {SEM_TXT_COMO_NEGATIVA!r}). Examples: {SEM_LABEL[:4]}")

print("🧭 Label schema of the images:"); print(df['schema'].value_counts().to_string())
amb = df[df['schema'] == 'AMBIGUA']
if len(amb):
    mes = r'(?:janeiro|fevereiro|marco|março|abril|maio|junho|julho|agosto|setembro|outubro|novembro|dezembro)-20\d\d'
    print(f"⚠️ {len(amb)} images contain class 1 but no class 2 (M. leonina of IL or O. flavescens of ML?): "
          f"{int(amb.c1.sum())} boxes of class 1; {int(amb.c0.gt(0).sum())} of them also have class 0")
    print(f"   with a Portuguese date in the name: {int(amb['image'].str.contains(mes, case=False, regex=True).sum())} | "
          f"others: {int((~amb['image'].str.contains(mes, case=False, regex=True)).sum())}")
    print("   Examples:", amb['image'].head(6).tolist())
    if AMBIGUAS == 'avisar':
        raise SystemExit("⏸️ Stopped: tell me which schema these images follow (IL_ORIGEM_DIRS, IL_NOME_REGEX / ML_NOME_REGEX, or AMBIGUAS = 'IL' / 'ML') and run again.")
    df.loc[df['schema'] == 'AMBIGUA', 'schema'] = AMBIGUAS
    print(f"   → assumed schema {AMBIGUAS} for them")

# counts in the OUTPUT schema (0 = A. australis, 1 = O. flavescens); M. leonina only exists in the IL schema (class 1)
il = df['schema'] == 'IL'
df['A'] = df['c0']
df['O'] = np.where(il, df['c2'], df['c1'])
df['leo'] = np.where(il, df['c1'], 0)
n_only_leo = int(((df['A'] + df['O'] == 0) & (df['leo'] > 0)).sum())
n_mixed_leo = int(((df['A'] + df['O'] > 0) & (df['leo'] > 0)).sum())
print(f"🚫 Images with ONLY M. leonina (excluded): {n_only_leo} | ⚠️ images with M. leonina and other species (leonina boxes removed): {n_mixed_leo} "
      f"| M. leonina boxes removed in total: {int(df.leo.sum())}")
df = df[~((df['A'] + df['O'] == 0) & (df['leo'] > 0))].reset_index(drop=True)
df['neg'] = ((df['A'] + df['O']) == 0).astype(int)
print(f"📷 {len(df)} images | boxes: A. australis = {df.A.sum()}, O. flavescens = {df.O.sum()} | negatives = {df.neg.sum()}")

In [ ]:
# ---- 2. flight from the file name ----
RE_DJI  = re.compile(r'(DJI)_?(\d{8})(\d{6})_(\d{4})(?:_([A-Z]))?')
RE_DATE = re.compile(r'(?<!\d)(20\d{2})[-_\.]?(0[1-9]|1[0-2])[-_\.]?(0[1-9]|[12]\d|3[01])(?!\d)')
MESES = {'janeiro': 1, 'fevereiro': 2, 'marco': 3, 'março': 3, 'abril': 4, 'maio': 5, 'junho': 6, 'julho': 7,
         'agosto': 8, 'setembro': 9, 'outubro': 10, 'novembro': 11, 'dezembro': 12}
RE_PT   = re.compile(r'(?<!\d)(\d{1,2})[-_ ](' + '|'.join(MESES) + r')[-_ ](20\d{2})(?:[-_ ](\d{1,2}))?(?!\d)', re.I)
RE_VOO  = re.compile(r'(?:voo|flight|fly|v)[-_]?(\d{1,3})(?!\d)', re.I)

def voo_do_nome(nome):
    m = RE_DJI.search(nome)
    if m:
        d, h, n, s = m.group(2), m.group(3), m.group(4), m.group(5) or ''
        return d, f"DJI_{d}{h}_{n}" + (f"_{s}" if s else ''), 'nome'
    m = RE_PT.search(nome)          # e.g. 1-marco-2022-2  (day-month-year[-flight number])
    if m:
        d = f"{m.group(3)}{MESES[m.group(2).lower()]:02d}{int(m.group(1)):02d}"
        return d, d, 'nome_data'      # trailing number (e.g. 11-julho-2022-3) is a photo index -> the whole day is one group
    m = RE_DATE.search(nome)
    if m:
        d = ''.join(m.groups())
        v = RE_VOO.search(nome[m.end():])
        return d, f"{d}" + (f"_voo{int(v.group(1))}" if v else ''), 'nome_data'
    return None, None, None

df[['data', 'voo', 'fonte']] = df['image'].apply(lambda n: pd.Series(voo_do_nome(n)))


In [ ]:
# ---- 3. manual override ----
if OVERRIDE_CSV:
    ov = pd.read_csv(OVERRIDE_CSV, sep=None, engine='python')
    for _, r in ov.iterrows():
        m = df['image'].str.startswith(str(r['prefixo']))
        df.loc[m, 'voo'] = str(r['voo']); df.loc[m, 'fonte'] = 'manual'
        d = RE_DATE.search(str(r['voo']))
        df.loc[m, 'data'] = ''.join(d.groups()) if d else df.loc[m, 'data']


In [ ]:
# ---- 4. EXIF date/time for the images still without a flight ----
from datetime import datetime
def exif_dt(path):
    try:
        ex = Image.open(path).getexif()
        v = ex.get_ifd(0x8769).get(0x9003) or ex.get(306)
        return datetime.strptime(str(v), '%Y:%m:%d %H:%M:%S') if v else None
    except Exception:
        return None

sem = df['voo'].isna()
if sem.any():
    dts = df.loc[sem, 'src_img'].apply(exif_dt)
    ok = dts.notna()
    if ok.any():
        t = pd.DataFrame({'idx': dts[ok].index, 'dt': dts[ok].values}).sort_values('dt').reset_index(drop=True)
        gap = t['dt'].diff() > pd.Timedelta(minutes=GAP_MIN)
        t['bloco'] = gap.cumsum()
        for b, g in t.groupby('bloco'):
            df.loc[g['idx'], 'voo'] = 'EXIF_' + g['dt'].min().strftime('%Y%m%d_%H%M')
            df.loc[g['idx'], 'data'] = g['dt'].min().strftime('%Y%m%d')
            df.loc[g['idx'], 'fonte'] = 'exif'
print("🏷️  Flight source (name / name_date / manual / exif):")
print(df['fonte'].value_counts(dropna=False).to_string())

In [ ]:
# ---- 5. visual similarity for the rest (works at the level of the ORIGINAL image: 'base') ----
def feat(path):
    im = Image.open(path).convert('RGB')
    th = np.asarray(im.resize((16, 16)), dtype=np.float32).ravel() / 255.0
    hsv = np.asarray(im.resize((128, 128)).convert('HSV'), dtype=np.float32)
    hs = np.concatenate([np.histogram(hsv[..., k], bins=16, range=(0, 255))[0] / (128 * 128) for k in range(3)])
    return np.concatenate([th, hs * 4])

sem = df['voo'].isna()
if sem.any():
    from sklearn.cluster import AgglomerativeClustering
    bases = df.groupby('base').first()                        # one image per original (augmented copies share it)
    F = {b: feat(p) for b, p in bases['src_img'].items()}
    conhecidos = df[~sem].groupby('base')['voo'].first()
    B_unk = [b for b in bases.index if b not in conhecidos.index]
    unk_voo = {}
    if len(conhecidos.unique()) >= 2:
        cent = {v: np.mean([F[b] for b in g.index], axis=0) for v, g in conhecidos.groupby(conhecidos)}
        # distance threshold: 95th percentile of the distance between a known image and its own flight centroid
        d_intra = [np.linalg.norm(F[b] - cent[v]) for b, v in conhecidos.items()]
        thr = float(np.percentile(d_intra, 95))
        # calibration: how often is the nearest centroid the true flight? (reliability of the visual assignment)
        acc = np.mean([min(cent, key=lambda v: np.linalg.norm(F[b] - cent[v])) == v for b, v in conhecidos.items()])
        print(f"🔎 Visual calibration on images with a known flight: nearest-flight accuracy = {acc:.1%} (threshold = {thr:.3f})")
        rest = []
        for b in B_unk:
            v = min(cent, key=lambda v: np.linalg.norm(F[b] - cent[v]))
            if np.linalg.norm(F[b] - cent[v]) <= thr:
                unk_voo[b] = v
            else:
                rest.append(b)
        if len(rest) >= 2:
            cl = AgglomerativeClustering(n_clusters=None, distance_threshold=thr * 1.5, linkage='average').fit(np.array([F[b] for b in rest]))
            for b, c in zip(rest, cl.labels_):
                unk_voo[b] = f'VISUAL_{c + 1:02d}'
        else:
            for b in rest:
                unk_voo[b] = 'VISUAL_01'
    else:
        k = min(N_VOOS_VISUAL, len(B_unk))
        cl = AgglomerativeClustering(n_clusters=k, linkage='ward').fit(np.array([F[b] for b in B_unk]))
        for b, c in zip(B_unk, cl.labels_):
            unk_voo[b] = f'VISUAL_{c + 1:02d}'
    m = df['voo'].isna()
    df.loc[m, 'voo'] = df.loc[m, 'base'].map(unk_voo)
    df.loc[m, 'fonte'] = 'visual'
    # an augmented copy must follow its original even if the original had a name-based flight
df['voo'] = df.groupby('base')['voo'].transform('first')
df['fonte'] = df.groupby('base')['fonte'].transform('first')
df['data'] = df.groupby('base')['data'].transform('first')

df['grupo'] = np.where((AGRUPAR_POR == 'data') & df['data'].notna(), 'DATA_' + df['data'].astype(str), df['voo'])
g = df.groupby('grupo').agg(imagens=('image', 'size'), A=('A', 'sum'), O=('O', 'sum'), neg=('neg', 'sum'),
                            fonte=('fonte', 'first'), data=('data', 'first')).sort_values('imagens', ascending=False)
print(f"\n✈️  {len(g)} groups ({AGRUPAR_POR}):")
print(g.to_string())
if AGRUPAR_POR == 'data':
    sd = df['data'].isna()
    print(f"📅 Dated images: {int((~sd).sum())} in {df.loc[~sd, 'data'].nunique()} different days | "
          f"WITHOUT a date (grouped by visual similarity, day not guaranteed): {int(sd.sum())}")
if len(g) < 6:
    print("⚠️ Few groups: an exact 70/15/15 split of images and boxes may not be possible.")

In [ ]:
# ======================================================
# CELL B. Assign whole groups to train / val / test
#   Minimises the deviation from the targets (70/15/15) in: images, A. australis boxes,
#   O. flavescens boxes and negative images. A group is never split.
# ======================================================
SPL = list(PROP)
V = g[['imagens', 'A', 'O', 'neg']].to_numpy(dtype=float)
TOT = V.sum(axis=0)
TGT = np.array([[PROP[s] * t for t in TOT] for s in SPL])
W = np.array([1.0, 2.0, 1.0, 0.5])        # weight: images, A. australis, O. flavescens, negatives

def perda(assign):
    S = np.zeros((len(SPL), 4))
    for k in range(len(SPL)):
        S[k] = V[assign == k].sum(axis=0)
    L = (W * ((S - TGT) / np.maximum(TOT, 1)) ** 2).sum()
    for m in (1, 2):                       # every split must contain boxes of both species
        if TOT[m] > 0:
            L += 10.0 * float((S[:, m] == 0).sum())
    return L

def otimizar(rng):
    G = len(V)
    a = np.zeros(G, dtype=int)
    S = np.zeros((len(SPL), 4))
    for i in rng.permutation(G):           # random greedy start
        k = int(np.argmax((TGT - S)[:, 0] + rng.normal(0, 0.05 * TGT[:, 0].max(), len(SPL))))
        a[i] = k; S[k] += V[i]
    best = perda(a)
    melhorou = True
    while melhorou:
        melhorou = False
        for i in range(G):                 # move one group
            for k in range(len(SPL)):
                if k == a[i]: continue
                old = a[i]; a[i] = k
                l = perda(a)
                if l < best - 1e-12: best = l; melhorou = True
                else: a[i] = old
        for i in range(G):                 # swap two groups
            for j in range(i + 1, G):
                if a[i] == a[j]: continue
                a[i], a[j] = a[j], a[i]
                l = perda(a)
                if l < best - 1e-12: best = l; melhorou = True
                else: a[i], a[j] = a[j], a[i]
    return a.copy(), best

rng = np.random.default_rng(SEED)
melhor, bl = None, np.inf
for r in range(60):
    a, l = otimizar(rng)
    if l < bl: melhor, bl = a, l
g['split'] = [SPL[k] for k in melhor]
df['split'] = df['grupo'].map(g['split'])

# ---- checks ----
assert df['split'].notna().all()
assert (df.groupby('grupo')['split'].nunique() == 1).all(), "A group ended up in more than one split"
if AGRUPAR_POR == 'data':
    assert (df.dropna(subset=['data']).groupby('data')['split'].nunique() == 1).all(), "A date ended up in more than one split"
assert (df.groupby('base')['split'].nunique() == 1).all(), "An original image and its copies are in different splits"

def resumo(d):
    return pd.Series({'Images': len(d), 'Positive': int((d.neg == 0).sum()), 'Negative': int(d.neg.sum()),
                      'A. australis (boxes)': int(d.A.sum()), 'O. flavescens (boxes)': int(d.O.sum()), 'Groups': d['grupo'].nunique()})
tab = pd.DataFrame({s: resumo(df[df.split == s]) for s in SPL}).T
tab.loc['Total'] = tab.sum()
pct = (tab.drop(columns='Groups').div(tab.loc['Total', tab.columns.drop('Groups')]) * 100).round(1)
print("\n📊 Result (counts):"); print(tab.to_string())
print("\n📊 Result (% of the total; target = " + " / ".join(f"{int(PROP[s]*100)}" for s in SPL) + "):"); print(pct.to_string())
print("\n📅 Dates present in more than one split (information; allowed if AGRUPAR_POR='voo'):")
dd = df.dropna(subset=['data']).groupby('data')['split'].agg(lambda x: ', '.join(sorted(set(x))))
print(dd[dd.str.contains(',')].to_string() if (dd.str.contains(',')).any() else '   none')

In [ ]:
# ======================================================
# CELL C. Copy to the new folders + save the tables
# ======================================================
os.makedirs(OUT_DIR, exist_ok=True)
df[['image', 'base', 'grupo', 'voo', 'data', 'fonte', 'split', 'A', 'O', 'neg']].to_csv(
    os.path.join(OUT_DIR, 'split_por_voo_imagens.csv'), index=False, encoding='utf-8-sig')
g.reset_index().to_csv(os.path.join(OUT_DIR, 'split_por_voo_grupos.csv'), index=False, encoding='utf-8-sig')
tab.to_csv(os.path.join(OUT_DIR, 'split_por_voo_resumo.csv'), encoding='utf-8-sig')

if COPIAR:
    for s in SPL:
        d = os.path.join(OUT_DIR, s)
        if os.path.exists(d):
            shutil.rmtree(d)
        os.makedirs(os.path.join(d, 'images')); os.makedirs(os.path.join(d, 'labels'))
    for _, r in df.iterrows():
        d = os.path.join(OUT_DIR, r['split'])
        shutil.copy(r['src_img'], os.path.join(d, 'images', r['image']))
        dst = os.path.join(d, 'labels', r['stem'] + '.txt')
        keep = []
        if not r['sem_txt']:
            for l in open(r['src_lbl']):
                p = l.split()
                if not p:
                    continue
                c = int(p[0])
                if r['schema'] == 'IL':
                    if c == 0: keep.append(l)
                    elif c == 2: keep.append('1 ' + ' '.join(p[1:]) + '\n')      # O. flavescens: 2 -> 1; class 1 (M. leonina) dropped
                elif c in (0, 1):
                    keep.append(l)
        open(dst, 'w').writelines(keep)
    for s in SPL:
        ni = len(os.listdir(os.path.join(OUT_DIR, s, 'images'))); nl = len(os.listdir(os.path.join(OUT_DIR, s, 'labels')))
        assert ni == nl == (df.split == s).sum(), f"Count mismatch in {s}"
    print(f"\n✅ Copied to {OUT_DIR}")
else:
    print("\nℹ️ COPIAR = False: nothing was copied")
print("   Review split_por_voo_imagens.csv (column 'fonte') before training, especially the rows with fonte = 'visual'.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# Dados do relatório
images_data = [
    {'Split': 'Train', 'Categoria': 'Imagens Positivas', 'Qtd': 970, 'Pct': 70.0},
    {'Split': 'Train', 'Categoria': 'Imagens Negativas', 'Qtd': 70, 'Pct': 70.0},
    {'Split': 'Val', 'Categoria': 'Imagens Positivas', 'Qtd': 208, 'Pct': 15.0},
    {'Split': 'Val', 'Categoria': 'Imagens Negativas', 'Qtd': 15, 'Pct': 15.0},
    {'Split': 'Test', 'Categoria': 'Imagens Positivas', 'Qtd': 208, 'Pct': 15.0},
    {'Split': 'Test', 'Categoria': 'Imagens Negativas', 'Qtd': 15, 'Pct': 15.0}
]

boxes_data = [
    {'Split': 'Train', 'Classe': 'A. australis (C0)', 'Qtd': 6011, 'Pct': 70.0},
    {'Split': 'Train', 'Classe': 'O. flavescens (C1)', 'Qtd': 10247, 'Pct': 70.0},
    {'Split': 'Val', 'Classe': 'A. australis (C0)', 'Qtd': 1288, 'Pct': 15.0},
    {'Split': 'Val', 'Classe': 'O. flavescens (C1)', 'Qtd': 2195, 'Pct': 15.0},
    {'Split': 'Test', 'Classe': 'A. australis (C0)', 'Qtd': 1287, 'Pct': 15.0},
    {'Split': 'Test', 'Classe': 'O. flavescens (C1)', 'Qtd': 2196, 'Pct': 15.0}
]

df_img_plot = pd.DataFrame(images_data)
df_box_plot = pd.DataFrame(boxes_data)

sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(16, 7.5))

# Painel 1: Distribuição Física de Imagens
ax1 = sns.barplot(
    x='Categoria',
    y='Qtd',
    hue='Split',
    data=df_img_plot,
    palette='viridis',
    ax=axes[0]
)
axes[0].set_title('Distribuição Física de Imagens por Split', fontsize=14, fontweight='bold', pad=15)
axes[0].set_xlabel('Categorias de Imagem', fontsize=12, labelpad=10)
axes[0].set_ylabel('Contagem Total (Imagens)', fontsize=12)
axes[0].legend(title='Splits', title_fontsize='11')

# Ajuste de limite do eixo Y para dar espaço às anotações no topo
axes[0].set_ylim(0, df_img_plot['Qtd'].max() * 1.15)

for p, (_, row) in zip(axes[0].patches, df_img_plot.iterrows()):
    height = p.get_height()
    if height > 0:
        axes[0].annotate(
            f'{int(height):,}\n({row["Pct"]:.1f}%)',
            (p.get_x() + p.get_width() / 2., height),
            ha='center',
            va='bottom',
            fontsize=9.5,
            fontweight='bold',
            xytext=(0, 5),
            textcoords='offset points'
        )

# Painel 2: Distribuição de Bounding Boxes
ax2 = sns.barplot(
    x='Classe',
    y='Qtd',
    hue='Split',
    data=df_box_plot,
    palette='viridis',
    ax=axes[1]
)
axes[1].set_title('Distribuição de Bounding Boxes por Classe', fontsize=14, fontweight='bold', pad=15)
axes[1].set_xlabel('Classes de Animais', fontsize=12, labelpad=10)
axes[1].set_ylabel('Contagem Total (Labels)', fontsize=12)
axes[1].legend(title='Splits', title_fontsize='11')

# Ajuste de limite do eixo Y para dar espaço às anotações no topo
axes[1].set_ylim(0, df_box_plot['Qtd'].max() * 1.15)

for p, (_, row) in zip(axes[1].patches, df_box_plot.iterrows()):
    height = p.get_height()
    if height > 0:
        axes[1].annotate(
            f'{int(height):,}\n({row["Pct"]:.1f}%)',
            (p.get_x() + p.get_width() / 2., height),
            ha='center',
            va='bottom',
            fontsize=9.5,
            fontweight='bold',
            xytext=(0, 5),
            textcoords='offset points'
        )

plt.suptitle('Avaliação do Dataset Final Reorganizado por Voo (70/15/15)', fontsize=16, fontweight='bold', y=0.98)
plt.tight_layout()
plt.show()

In [ ]:
# ======================================================
# Install dependency (Ultralytics/YOLO)
# ======================================================
!pip install -q ultralytics

# ======================================================
# 7b. Imports and paths of the NEW split (70/15/15, whole days in one split only)
#     (run once; if the notebook already has these imports, nothing breaks)
# ======================================================
import os, re, glob, json, shutil
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import ultralytics
from ultralytics import YOLO

In [ ]:
MODELO_DIR = '/content/drive/MyDrive/Colab Notebooks/YOLO_IL_Negativo_ML/YOLO_v26_Negativo/Modelo_Unico_IL_ML/Modelo_Julian_Mathias'
split_dir  = os.path.join(MODELO_DIR, 'Split_70_15_15_por_voo_2')       # train/ val/ test/ (each with images/ and labels/)
test_dir   = os.path.join(split_dir, 'test')
base_path  = MODELO_DIR                                              # kept for compatibility with the old cells

# Sanity check: folders, image/label counts and class IDs (must be only 0 = A_AUSTRALIS and 1 = O_FLAVESCENS)
EXTS = ('.jpg', '.jpeg', '.png')
ids_found, resumo = {}, []
for s in ['train', 'val', 'test']:
    img_d, lbl_d = os.path.join(split_dir, s, 'images'), os.path.join(split_dir, s, 'labels')
    assert os.path.isdir(img_d) and os.path.isdir(lbl_d), f"❌ Missing folder: {s}"
    imgs = [f for f in os.listdir(img_d) if f.lower().endswith(EXTS)]
    lbls = [f for f in os.listdir(lbl_d) if f.endswith('.txt')]
    assert len(imgs) == len(lbls), f"❌ {s}: {len(imgs)} images vs {len(lbls)} labels"
    n = {0: 0, 1: 0}
    for f in lbls:
        for line in open(os.path.join(lbl_d, f)):
            p = line.split()
            if p:
                c = int(p[0]); ids_found[c] = ids_found.get(c, 0) + 1
                if c in n: n[c] += 1
    resumo.append({'split': s, 'images': len(imgs), 'A_AUSTRALIS': n[0], 'O_FLAVESCENS': n[1]})
assert set(ids_found) <= {0, 1}, f"❌ Unexpected class IDs: {ids_found}"
print(pd.DataFrame(resumo).to_string(index=False))
print("✅ Split verified")

In [ ]:
# ======================================================
# 8. Configuration, File Paths, and Dataset YAML
# ======================================================
RESULTS_DIR = os.path.join(MODELO_DIR, 'Resultados_Split_70_15_15_por_voo')   # new folder: nothing from previous runs is overwritten
os.makedirs(RESULTS_DIR, exist_ok=True)

CLASS_NAMES = ['A_AUSTRALIS', 'O_FLAVESCENS']
EPOCHS = 100
IMGSZ = 640
EVAL_TREINO_POR_EPOCA = True   # Compute training mAP per epoch (increases runtime)

# Primary YAML dataset configuration
yaml_file_name = os.path.join(RESULTS_DIR, 'yolov26_config.yaml')
yaml_content = f"""
path: {split_dir}
train: train/images
val: val/images
test: test/images
nc: 2
names:
  0: A_AUSTRALIS
  1: O_FLAVESCENS
"""
with open(yaml_file_name, 'w') as f:
    f.write(yaml_content.strip())

# Auxiliary YAML: redirects validation to training split to evaluate training mAP
yaml_train_eval = os.path.join(RESULTS_DIR, 'yolov26_config_train_eval.yaml')
with open(yaml_train_eval, 'w') as f:
    f.write(yaml_content.replace('val: val/images', 'val: train/images').strip())

print(f"✅ YAML configurations created\n📁 Destination directory: {RESULTS_DIR}")

In [ ]:
# ======================================================
# 9. Model Training (YOLO26m)
# ======================================================
model = YOLO('yolo26m.pt')

print(f"\n🚀 Initialising training process (Ultralytics {ultralytics.__version__})...")

results = model.train(
    data=yaml_file_name,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=16,
    patience = 10,
    mosaic = 0.0,
    scale = 0.0,
    seed=42,
    deterministic=True,
    project=RESULTS_DIR,
    name='treino',
    exist_ok=True,
    save_period=1 if EVAL_TREINO_POR_EPOCA else -1,
    plots=True
)

run_dir = str(model.trainer.save_dir)
best_pt = os.path.join(run_dir, 'weights', 'best.pt')
print(f"\n✅ Training complete!\n📁 Output saved to: {run_dir}")

In [ ]:
# ======================================================
# 10. Training vs Validation Performance Curves per Epoch
# ======================================================
res = pd.read_csv(os.path.join(run_dir, 'results.csv'))
res.columns = res.columns.str.strip()
epochs = res['epoch'].astype(int).values

# ------------------------------------------------------
# Training mAP per epoch (evaluates each checkpoint on training set)
# ------------------------------------------------------
train_map50, train_map5095 = None, None

if EVAL_TREINO_POR_EPOCA:
    ckpts = sorted(
        glob.glob(os.path.join(run_dir, 'weights', 'epoch*.pt')),
        key=lambda p: int(re.findall(r'epoch(\d+)\.pt', p)[0])
    )
    train_map50, train_map5095 = [], []

    for p in ckpts:
        m = YOLO(p).val(
            data=yaml_train_eval,
            split='val',
            imgsz=IMGSZ,
            plots=False,
            verbose=False,
            project=RESULTS_DIR,
            name='tmp_eval',
            exist_ok=True
        )
        train_map50.append(m.box.map50)
        train_map5095.append(m.box.map)

    # The i-th checkpoint corresponds to the i-th row in results.csv
    n = min(len(ckpts), len(res))
    res.loc[:n - 1, 'train_mAP50'] = train_map50[:n]
    res.loc[:n - 1, 'train_mAP50-95'] = train_map5095[:n]

    # Free storage space in Google Drive
    for p in ckpts:
        os.remove(p)
    shutil.rmtree(os.path.join(RESULTS_DIR, 'tmp_eval'), ignore_errors=True)

res.to_csv(os.path.join(RESULTS_DIR, 'metrics_per_epoch.csv'), index=False)

# ------------------------------------------------------
# Precision curves (mAP50 and mAP50-95)
# ------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 5.5))

ax.plot(epochs, res['metrics/mAP50(B)'], '-o', ms=4, color='#1f77b4', label='mAP50 – Validation')
ax.plot(epochs, res['metrics/mAP50-95(B)'], '-o', ms=4, color='#ff7f0e', label='mAP50-95 – Validation')

if EVAL_TREINO_POR_EPOCA:
    ax.plot(epochs, res['train_mAP50'], '--s', ms=4, color='#1f77b4', alpha=0.7, label='mAP50 – Training')
    ax.plot(epochs, res['train_mAP50-95'], '--s', ms=4, color='#ff7f0e', alpha=0.7, label='mAP50-95 – Training')

ax.set_xlabel('Epoch')
ax.set_ylabel('mAP')
ax.set_title('mAP across epochs: training vs validation')
ax.set_ylim(0, 1)
ax.grid(linestyle='--', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'mAP_curve_per_epoch.png'), dpi=300, bbox_inches='tight')
plt.show()

# ------------------------------------------------------
# Training vs Validation loss curves (overfitting diagnosis)
# ------------------------------------------------------
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(epochs, res['train/box_loss'], '-o', ms=4, label='box_loss – Training')
ax.plot(epochs, res['val/box_loss'], '-o', ms=4, label='box_loss – Validation')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.set_title('Box loss across epochs')
ax.grid(linestyle='--', alpha=0.5)
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'loss_curve_per_epoch.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# ======================================================
# 11. Test Set Evaluation
# ======================================================
model_best = YOLO(best_pt)

# ------------------------------------------------------
# Confidence threshold determined on VALIDATION split (maximum macro F1-score)
# and applied directly to the test set without tuning on test data
# ------------------------------------------------------
mv = model_best.val(
    data=yaml_file_name,
    split='val',
    imgsz=IMGSZ,
    plots=False,
    verbose=False,
    project=RESULTS_DIR,
    name='tmp_val',
    exist_ok=True
)

f1_mean_val = mv.box.f1_curve.mean(0)
from ultralytics.utils.metrics import smooth   # Standard smoothing function employed by Ultralytics

best_conf = float(mv.box.px[smooth(f1_mean_val, 0.1).argmax()])
shutil.rmtree(os.path.join(RESULTS_DIR, 'tmp_val'), ignore_errors=True)
print(f"🎯 Selected confidence threshold (maximum validation F1-score): {best_conf:.3f}")

# ------------------------------------------------------
# Test Set Inference and Evaluation
# ------------------------------------------------------
mt = model_best.val(
    data=yaml_file_name,
    split='test',
    imgsz=IMGSZ,
    plots=True,                                # Generates confusion matrix, PR curves, F1 curves, etc.
    project=RESULTS_DIR,
    name='teste',
    exist_ok=True
)

b = mt.box
idx = int(np.abs(b.px - best_conf).argmin())

# Confusion matrix at the validation-chosen threshold (the default val() uses a very low conf and inflates
# background false positives). Second val() ONLY for the matrix; mAP and PR curves come from the call above.
model_best.val(
    data=yaml_file_name,
    split='test',
    imgsz=IMGSZ,
    conf=best_conf,
    plots=True,
    project=RESULTS_DIR,
    name='teste_matriz_confusao_conf_otima',
    exist_ok=True,
    verbose=False
)
print(f"🧮 Confusion matrix at conf = {best_conf:.3f} saved to: {os.path.join(RESULTS_DIR, 'teste_matriz_confusao_conf_otima')}")

# ------------------------------------------------------
# Performance Metrics Table (Precision, Recall, and F1 at validation threshold)
# ------------------------------------------------------
t_img = os.path.join(test_dir, 'images')

rows = []
for k, c in enumerate(b.ap_class_index):
    rows.append({
        'Class': CLASS_NAMES[c],
        'mAP50': b.ap50[k],
        'mAP50-95': b.ap[k],
        'Precision': b.p_curve[k, idx],
        'Recall': b.r_curve[k, idx],
        'F1-Score': b.f1_curve[k, idx],
    })

# Compute macro-average row ('all')
rows.insert(0, {
    'Class': 'all',
    'mAP50': b.map50,
    'mAP50-95': b.map,
    'Precision': np.mean([r['Precision'] for r in rows]),
    'Recall': np.mean([r['Recall'] for r in rows]),
    'F1-Score': np.mean([r['F1-Score'] for r in rows]),
})

df_test = pd.DataFrame(rows)
df_test.to_csv(os.path.join(RESULTS_DIR, 'metricas_teste.csv'), index=False, encoding='utf-8-sig')

info = {
    'confidence_threshold': best_conf,
    'num_test_images': len(os.listdir(t_img)),
}
with open(os.path.join(RESULTS_DIR, 'metricas_teste_info.json'), 'w') as f:
    json.dump(info, f, indent=2, ensure_ascii=False)

print("\n" + "=" * 70)
print("📈 TEST SET PERFORMANCE METRICS")
print("=" * 70)
print(df_test.round(4).to_string(index=False))

# ------------------------------------------------------
# Precision–Recall Curve Visualisation
# ------------------------------------------------------
fig, ax = plt.subplots(figsize=(7, 6))
colours = ['#264653', '#e76f51']

for k, c in enumerate(b.ap_class_index):
    ax.plot(
        b.px,
        b.prec_values[k],
        color=colours[c],
        lw=2,
        label=f'{CLASS_NAMES[c]} (AP50 = {b.ap50[k]:.3f})'
    )

ax.plot(
    b.px,
    b.prec_values.mean(0),
    color='black',
    lw=3,
    label=f'All classes (mAP50 = {b.map50:.3f})'
)

ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.05)
ax.set_title('Precision–Recall Curve (Independent Test Set)')
ax.grid(linestyle='--', alpha=0.5)
ax.legend(loc='lower left')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'curva_precision_recall_teste.png'), dpi=300, bbox_inches='tight')
plt.show()

print(f"\n✅ All outputs successfully saved to: {RESULTS_DIR}")


In [ ]:
# ======================================================
# 12. Visual Inference on Independent Test Set
# ======================================================
# Fallback to default 0.4 if best_conf was not previously defined in runtime
CONF_PRED = best_conf if 'best_conf' in locals() else 0.4
IOU_PRED = 0.6

model_best = YOLO(best_pt)
pred_dir = os.path.join(RESULTS_DIR, 'predicoes_teste')
os.makedirs(pred_dir, exist_ok=True)

print(f"\n🎯 Exporting test set predictions to: {pred_dir}")

rows = []
stream = model_best.predict(
    source=os.path.join(test_dir, 'images'),  # Includes negative control samples
    conf=CONF_PRED,
    iou=IOU_PRED,
    imgsz=IMGSZ,
    save=True,
    project=RESULTS_DIR,
    name='predicoes_teste',
    exist_ok=True,
    stream=True,
    verbose=False
)

for r in stream:
    cls = r.boxes.cls.cpu().numpy().astype(int)
    rows.append({
        'image': os.path.basename(r.path),
        'A_AUSTRALIS': int((cls == 0).sum()),
        'O_FLAVESCENS': int((cls == 1).sum())
    })

# Export frame-level detections (verifies false positive rates on negative images)
df_pred = pd.DataFrame(rows)
df_pred.to_csv(os.path.join(pred_dir, 'deteccoes_por_imagem_teste.csv'), index=False)

print(f"\n✅ {len(df_pred)} images successfully processed")
print(f"📸 Annotated test images saved to: {pred_dir}")

In [ ]:
# ======================================================
# 13. Checkpoint Verification
# ======================================================
import os

# Fallback se a variável 'best_pt' não estiver na memória (ex: treino foi interrompido)
if 'best_pt' in locals():
    best_model_path = best_pt
else:
    # Define o caminho padrão baseado nas configurações anteriores
    RESULTS_DIR = os.path.join(MODELO_DIR, 'Resultados_Split_70_15_15_por_voo')
    best_model_path = os.path.join(RESULTS_DIR, 'treino', 'weights', 'best.pt')

assert os.path.exists(best_model_path), f"❌ Checkpoint best.pt not found: {best_model_path}"
print(f"✅ Optimal checkpoint verified: {best_model_path}")

### **YOLO26 - 1280 x 1280**

In [ ]:
# ======================================================
# CELL 1. Clean REVIS IL (1280x1280): drop M_LEONINA (class 1) and remap O_FLAVESCENS (2 -> 1)
#         Saves cleaned images & labels into the unified folder ready to receive REVIS ML.
# ======================================================
import os
import shutil

SRC_IMG_IL = '/content/drive/MyDrive/Colab Notebooks/YOLO_IL_Negativo_ML/YOLO_v26_Negativo/Modelo_Unico_IL_ML/Modelo_Julian_Mathias/REVIS_IL_1280x1280/train/images'
SRC_LBL_IL = '/content/drive/MyDrive/Colab Notebooks/YOLO_IL_Negativo_ML/YOLO_v26_Negativo/Modelo_Unico_IL_ML/Modelo_Julian_Mathias/REVIS_IL_1280x1280/train/labels'

# Folder where cleaned REVIS IL will be saved (and where you will move/copy REVIS ML 1280x1280 next)
UNIFIED_1280_DIR = '/content/drive/MyDrive/Colab Notebooks/YOLO_IL_Negativo_ML/YOLO_v26_Negativo/Modelo_Unico_IL_ML/Modelo_Julian_Mathias/Dataset_Unificado_1280x1280'
DST_IMG = os.path.join(UNIFIED_1280_DIR, 'images')
DST_LBL = os.path.join(UNIFIED_1280_DIR, 'labels')

os.makedirs(DST_IMG, exist_ok=True)
os.makedirs(DST_LBL, exist_ok=True)

EXTS = ('.jpg', '.jpeg', '.png')
stats = {'copied': 0, 'excluded_only_leonina': 0, 'leonina_boxes_removed': 0, 'aust_boxes': 0, 'flav_boxes': 0, 'negatives': 0}

for fname in sorted(os.listdir(SRC_IMG_IL)):
    if not fname.lower().endswith(EXTS):
        continue
    stem = os.path.splitext(fname)[0]
    src_img_path = os.path.join(SRC_IMG_IL, fname)
    src_lbl_path = os.path.join(SRC_LBL_IL, stem + '.txt')

    if not os.path.exists(src_lbl_path):
        print(f"⚠️ Label missing for {fname} (skipped)")
        continue

    raw_lines = [l.strip() for l in open(src_lbl_path) if l.strip()]

    # Original IL schema: 0 = A_AUSTRALIS, 1 = M_LEONINA, 2 = O_FLAVESCENS
    c0 = [l for l in raw_lines if int(l.split()[0]) == 0]
    c1 = [l for l in raw_lines if int(l.split()[0]) == 1]
    c2 = [l for l in raw_lines if int(l.split()[0]) == 2]

    stats['leonina_boxes_removed'] += len(c1)

    # Exclude images that had ONLY M_LEONINA (and no A_AUSTRALIS or O_FLAVESCENS)
    if len(raw_lines) > 0 and len(c0) == 0 and len(c2) == 0 and len(c1) > 0:
        stats['excluded_only_leonina'] += 1
        continue

    # Build remapped label lines (0 -> 0, 2 -> 1, 1 dropped)
    new_lines = []
    for l in raw_lines:
        parts = l.split()
        cid = int(parts[0])
        if cid == 0:
            new_lines.append('0 ' + ' '.join(parts[1:]) + '\n')
            stats['aust_boxes'] += 1
        elif cid == 2:
            new_lines.append('1 ' + ' '.join(parts[1:]) + '\n')
            stats['flav_boxes'] += 1

    if len(new_lines) == 0:
        stats['negatives'] += 1

    shutil.copy(src_img_path, os.path.join(DST_IMG, fname))
    with open(os.path.join(DST_LBL, stem + '.txt'), 'w') as out_f:
        out_f.writelines(new_lines)
    stats['copied'] += 1

print("✅ REVIS IL (1280x1280) cleaned and saved!")
print(f"   Destination images : {DST_IMG}")
print(f"   Destination labels : {DST_LBL}")
print(f"   Images saved       : {stats['copied']} (including {stats['negatives']} negatives)")
print(f"   Excluded (only M. leonina) : {stats['excluded_only_leonina']}")
print(f"   M. leonina boxes removed   : {stats['leonina_boxes_removed']}")
print(f"   Retained boxes     : A. australis = {stats['aust_boxes']} | O. flavescens = {stats['flav_boxes']}")
print("\n👉 Next step: Move/copy your REVIS ML (1280x1280) images and labels into the two folders above, then run Cell 2.")

In [ ]:
# ======================================================
# CELL 2. Mirror exact 70/15/15 flight/date split from previous experiment into 1280x1280
#         Guarantees 100% identical train/val/test image allocation for fair resolution comparison.
# ======================================================
import os
import re
import shutil
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

MODELO_DIR = '/content/drive/MyDrive/Colab Notebooks/YOLO_IL_Negativo_ML/YOLO_v26_Negativo/Modelo_Unico_IL_ML/Modelo_Julian_Mathias'

# Reference CSV from the previous split (640x640)
REF_SPLIT_DIR = os.path.join(MODELO_DIR, 'Split_70_15_15_por_voo_2')
REF_CSV = os.path.join(REF_SPLIT_DIR, 'split_por_voo_imagens.csv')

# Unified 1280x1280 source (REVIS IL cleaned + REVIS ML moved)
UNIFIED_1280_DIR = os.path.join(MODELO_DIR, 'Dataset_Unificado_1280x1280')
SRC_IMG = os.path.join(UNIFIED_1280_DIR, 'images')
SRC_LBL = os.path.join(UNIFIED_1280_DIR, 'labels')

# Output directory requested
OUT_DIR = os.path.join(MODELO_DIR, 'Split_70_15_15_por_voo_1280x1280')

EXTS = ('.jpg', '.jpeg', '.png')
RF = re.compile(r'[_\.]?(jpg|jpeg|png)\.rf\.[0-9a-f]+$', re.I)

assert os.path.exists(REF_CSV), f"❌ Reference split CSV not found: {REF_CSV}"
ref_df = pd.read_csv(REF_CSV)

# Map both exact filename and base filename (without Roboflow hash) -> split ('train', 'val', 'test')
exact_to_split = dict(zip(ref_df['image'], ref_df['split']))
base_to_split = dict(zip(ref_df['base'], ref_df['split']))
base_to_meta = ref_df.groupby('base')[['grupo', 'voo', 'data', 'fonte']].first().to_dict('index')

# Prepare output folders
for s in ['train', 'val', 'test']:
    d = os.path.join(OUT_DIR, s)
    if os.path.exists(d):
        shutil.rmtree(d)
    os.makedirs(os.path.join(d, 'images'), exist_ok=True)
    os.makedirs(os.path.join(d, 'labels'), exist_ok=True)

records = []
unmatched = []

for fname in sorted(os.listdir(SRC_IMG)):
    if not fname.lower().endswith(EXTS):
        continue
    stem = os.path.splitext(fname)[0]
    base = RF.sub('', stem)
    lbl_path = os.path.join(SRC_LBL, stem + '.txt')
    assert os.path.exists(lbl_path), f"❌ Label file missing for {fname}"

    # Match split from reference experiment
    if fname in exact_to_split:
        sp = exact_to_split[fname]
    elif base in base_to_split:
        sp = base_to_split[base]
    else:
        unmatched.append(fname)
        continue

    # Verify classes (must be strictly 0 and 1) and count boxes
    c0 = c1 = 0
    clean_lines = []
    for line in open(lbl_path):
        p = line.split()
        if not p:
            continue
        cid = int(p[0])
        assert cid in (0, 1), f"❌ Unexpected class {cid} in {lbl_path}"
        if cid == 0:
            c0 += 1
        else:
            c1 += 1
        clean_lines.append(line)

    # Copy image and label to target split
    shutil.copy(os.path.join(SRC_IMG, fname), os.path.join(OUT_DIR, sp, 'images', fname))
    with open(os.path.join(OUT_DIR, sp, 'labels', stem + '.txt'), 'w') as out_f:
        out_f.writelines(clean_lines)

    meta = base_to_meta.get(base, {'grupo': 'unknown', 'voo': 'unknown', 'data': None, 'fonte': 'mirrored'})
    records.append({
        'image': fname,
        'base': base,
        'grupo': meta['grupo'],
        'voo': meta['voo'],
        'data': meta['data'],
        'fonte': meta['fonte'],
        'split': sp,
        'A': c0,
        'O': c1,
        'neg': int((c0 + c1) == 0)
    })

if unmatched:
    print(f"⚠️ {len(unmatched)} images were not found in the previous split CSV (skipped): {unmatched[:5]}")

df_1280 = pd.DataFrame(records)
df_1280.to_csv(os.path.join(OUT_DIR, 'split_por_voo_imagens.csv'), index=False, encoding='utf-8-sig')

# Summary table
def resumo(d):
    return pd.Series({
        'Images': len(d),
        'Positive': int((d['neg'] == 0).sum()),
        'Negative': int(d['neg'].sum()),
        'A. australis (boxes)': int(d['A'].sum()),
        'O. flavescens (boxes)': int(d['O'].sum()),
        'Groups': d['grupo'].nunique()
    })

SPL = ['train', 'val', 'test']
tab = pd.DataFrame({s: resumo(df_1280[df_1280['split'] == s]) for s in SPL}).T
tab.loc['Total'] = tab.sum()
tab.to_csv(os.path.join(OUT_DIR, 'split_por_voo_resumo.csv'), encoding='utf-8-sig')

pct = (tab.drop(columns='Groups').div(tab.loc['Total', tab.columns.drop('Groups')]) * 100).round(1)
print("\n📊 Mirrored Split 1280x1280 (counts):")
print(tab.to_string())
print("\n📊 Mirrored Split 1280x1280 (%):")
print(pct.to_string())

# Verify file integrity across splits
for s in SPL:
    ni = len(os.listdir(os.path.join(OUT_DIR, s, 'images')))
    nl = len(os.listdir(os.path.join(OUT_DIR, s, 'labels')))
    assert ni == nl == (df_1280['split'] == s).sum(), f"❌ File count mismatch in {s}"

print(f"\n✅ Mirrored 1280x1280 split successfully created at: {OUT_DIR}")

In [ ]:
# ======================================================
# CELL 3. Training, Curves, Evaluation, and Inference (YOLO26m @ 1280x1280)
# ======================================================
!pip install -q ultralytics

import os
import re
import glob
import json
import shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ultralytics
from ultralytics import YOLO
from ultralytics.utils.metrics import smooth

MODELO_DIR = '/content/drive/MyDrive/Colab Notebooks/YOLO_IL_Negativo_ML/YOLO_v26_Negativo/Modelo_Unico_IL_ML/Modelo_Julian_Mathias'
split_dir = os.path.join(MODELO_DIR, 'Split_70_15_15_por_voo_1280x1280')
test_dir = os.path.join(split_dir, 'test')

RESULTS_DIR = os.path.join(MODELO_DIR, 'Resultados_Split_70_15_15_por_voo_1280x1280')
os.makedirs(RESULTS_DIR, exist_ok=True)

CLASS_NAMES = ['A_AUSTRALIS', 'O_FLAVESCENS']
EPOCHS = 50
IMGSZ = 1280
BATCH = 4                      # Adjusted from 4 to fit 1280x1280 resolution safely in T4 GPU VRAM
EVAL_TREINO_POR_EPOCA = True

In [ ]:
# 1. Dataset YAML configurations
yaml_file_name = os.path.join(RESULTS_DIR, 'yolov26_config_1280.yaml')
yaml_content = f"""
path: {split_dir}
train: train/images
val: val/images
test: test/images
nc: 2
names:
  0: A_AUSTRALIS
  1: O_FLAVESCENS
"""
with open(yaml_file_name, 'w') as f:
    f.write(yaml_content.strip())

yaml_train_eval = os.path.join(RESULTS_DIR, 'yolov26_config_1280_train_eval.yaml')
with open(yaml_train_eval, 'w') as f:
    f.write(yaml_content.replace('val: val/images', 'val: train/images').strip())


In [ ]:
# 2. Model Training
model = YOLO('yolo26m.pt')
print(f"\n🚀 Initialising training at {IMGSZ}x{IMGSZ} (Ultralytics {ultralytics.__version__})...")

results = model.train(
    data=yaml_file_name,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=10,
    mosaic=0.0,
    scale=0.0,
    seed=42,
    deterministic=True,
    project=RESULTS_DIR,
    name='treino',
    exist_ok=True,
    save_period=1 if EVAL_TREINO_POR_EPOCA else -1,
    plots=True
)

run_dir = str(model.trainer.save_dir)
best_pt = os.path.join(run_dir, 'weights', 'best.pt')
print(f"\n✅ Training complete!\n📁 Output saved to: {run_dir}")


In [ ]:
# 3. Training vs Validation Curves per Epoch
res = pd.read_csv(os.path.join(run_dir, 'results.csv'))
res.columns = res.columns.str.strip()
epochs = res['epoch'].astype(int).values

if EVAL_TREINO_POR_EPOCA:
    ckpts = sorted(
        glob.glob(os.path.join(run_dir, 'weights', 'epoch*.pt')),
        key=lambda p: int(re.findall(r'epoch(\d+)\.pt', p)[0])
    )
    train_map50, train_map5095 = [], []

    for p in ckpts:
        m = YOLO(p).val(
            data=yaml_train_eval,
            split='val',
            imgsz=IMGSZ,
            plots=False,
            verbose=False,
            project=RESULTS_DIR,
            name='tmp_eval',
            exist_ok=True
        )
        train_map50.append(m.box.map50)
        train_map5095.append(m.box.map)

    n = min(len(ckpts), len(res))
    res.loc[:n - 1, 'train_mAP50'] = train_map50[:n]
    res.loc[:n - 1, 'train_mAP50-95'] = train_map5095[:n]

    for p in ckpts:
        os.remove(p)
    shutil.rmtree(os.path.join(RESULTS_DIR, 'tmp_eval'), ignore_errors=True)

res.to_csv(os.path.join(RESULTS_DIR, 'metrics_per_epoch.csv'), index=False)

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.plot(epochs, res['metrics/mAP50(B)'], '-o', ms=4, color='#1f77b4', label='mAP50 – Validation')
ax.plot(epochs, res['metrics/mAP50-95(B)'], '-o', ms=4, color='#ff7f0e', label='mAP50-95 – Validation')
if EVAL_TREINO_POR_EPOCA:
    ax.plot(epochs, res['train_mAP50'], '--s', ms=4, color='#1f77b4', alpha=0.7, label='mAP50 – Training')
    ax.plot(epochs, res['train_mAP50-95'], '--s', ms=4, color='#ff7f0e', alpha=0.7, label='mAP50-95 – Training')
ax.set_xlabel('Epoch'); ax.set_ylabel('mAP')
ax.set_title(f'mAP across epochs ({IMGSZ}x{IMGSZ}): training vs validation')
ax.set_ylim(0, 1); ax.grid(linestyle='--', alpha=0.5); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'mAP_curve_per_epoch.png'), dpi=300, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(epochs, res['train/box_loss'], '-o', ms=4, label='box_loss – Training')
ax.plot(epochs, res['val/box_loss'], '-o', ms=4, label='box_loss – Validation')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title(f'Box loss across epochs ({IMGSZ}x{IMGSZ})')
ax.grid(linestyle='--', alpha=0.5); ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'loss_curve_per_epoch.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 4. Test Set Evaluation
model_best = YOLO(best_pt)

mv = model_best.val(
    data=yaml_file_name,
    split='val',
    imgsz=IMGSZ,
    plots=False,
    verbose=False,
    project=RESULTS_DIR,
    name='tmp_val',
    exist_ok=True
)
best_conf = float(mv.box.px[smooth(mv.box.f1_curve.mean(0), 0.1).argmax()])
shutil.rmtree(os.path.join(RESULTS_DIR, 'tmp_val'), ignore_errors=True)
print(f"🎯 Selected confidence threshold (maximum validation F1-score): {best_conf:.3f}")

mt = model_best.val(
    data=yaml_file_name,
    split='test',
    imgsz=IMGSZ,
    plots=True,
    project=RESULTS_DIR,
    name='teste',
    exist_ok=True
)
b = mt.box
idx = int(np.abs(b.px - best_conf).argmin())

model_best.val(
    data=yaml_file_name,
    split='test',
    imgsz=IMGSZ,
    conf=best_conf,
    plots=True,
    project=RESULTS_DIR,
    name='teste_matriz_confusao_conf_otima',
    exist_ok=True,
    verbose=False
)

rows = []
for k, c in enumerate(b.ap_class_index):
    rows.append({
        'Class': CLASS_NAMES[c],
        'mAP50': b.ap50[k],
        'mAP50-95': b.ap[k],
        'Precision': b.p_curve[k, idx],
        'Recall': b.r_curve[k, idx],
        'F1-Score': b.f1_curve[k, idx],
    })
rows.insert(0, {
    'Class': 'all',
    'mAP50': b.map50,
    'mAP50-95': b.map,
    'Precision': np.mean([r['Precision'] for r in rows]),
    'Recall': np.mean([r['Recall'] for r in rows]),
    'F1-Score': np.mean([r['F1-Score'] for r in rows]),
})

df_test = pd.DataFrame(rows)
df_test.to_csv(os.path.join(RESULTS_DIR, 'metricas_teste.csv'), index=False, encoding='utf-8-sig')
with open(os.path.join(RESULTS_DIR, 'metricas_teste_info.json'), 'w') as f:
    json.dump({'imgsz': IMGSZ, 'confidence_threshold': best_conf, 'num_test_images': len(os.listdir(os.path.join(test_dir, 'images')))}, f, indent=2)

print("\n" + "=" * 70)
print(f"📈 TEST SET PERFORMANCE METRICS ({IMGSZ}x{IMGSZ})")
print("=" * 70)
print(df_test.round(4).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 6))
colours = ['#264653', '#e76f51']
for k, c in enumerate(b.ap_class_index):
    ax.plot(b.px, b.prec_values[k], color=colours[c], lw=2, label=f'{CLASS_NAMES[c]} (AP50 = {b.ap50[k]:.3f})')
ax.plot(b.px, b.prec_values.mean(0), color='black', lw=3, label=f'All classes (mAP50 = {b.map50:.3f})')
ax.set_xlabel('Recall'); ax.set_ylabel('Precision')
ax.set_xlim(0, 1); ax.set_ylim(0, 1.05)
ax.set_title(f'Precision–Recall Curve (Test Set @ {IMGSZ}x{IMGSZ})')
ax.grid(linestyle='--', alpha=0.5); ax.legend(loc='lower left')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'curva_precision_recall_teste.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 5. Visual Inference on Independent Test Set
pred_dir = os.path.join(RESULTS_DIR, 'predicoes_teste')
os.makedirs(pred_dir, exist_ok=True)
pred_rows = []
stream = model_best.predict(
    source=os.path.join(test_dir, 'images'),
    conf=best_conf,
    iou=0.6,
    imgsz=IMGSZ,
    save=True,
    project=RESULTS_DIR,
    name='predicoes_teste',
    exist_ok=True,
    stream=True,
    verbose=False
)
for r in stream:
    cls = r.boxes.cls.cpu().numpy().astype(int)
    pred_rows.append({
        'image': os.path.basename(r.path),
        'A_AUSTRALIS': int((cls == 0).sum()),
        'O_FLAVESCENS': int((cls == 1).sum())
    })
pd.DataFrame(pred_rows).to_csv(os.path.join(pred_dir, 'deteccoes_por_imagem_teste.csv'), index=False)
print(f"\n✅ All 1280x1280 outputs saved to: {RESULTS_DIR}")